# WP3 — Endpoint construction starter (Student 1)

**What this notebook does.** Shows the WP3 pipeline pattern: read WP2 preprocessing outputs + hormone data (or the synthetic equivalent), produce `endpoint/cycle_boundaries.parquet` and refresh `labels.parquet` using the tiered-confidence rule (pipeline contract §4).

**Tiered-confidence rule** (subject to supervisor confirmation of Option 1 from the endpoint discussion):
- **gold (1.0):** LH surge detected AND progesterone-metabolite rise confirms ovulation (≥2 post-surge days elevated). Round 2 only.
- **silver (0.7):** LH surge detected, no progesterone data available. Round 1.
- **bronze (0.3):** within ±2 days of a detected LH surge (peri-ovulatory ambiguity).
- **charcoal (0.1):** LH surge without progesterone-rise confirmation in Round 2 (possibly anovulatory).

**Outputs.**
- `endpoint/cycle_boundaries.parquet`
- `labels.parquet` (refreshed — overwrites synthetic-generator output)


In [ ]:
import sys, os, subprocess
from pathlib import Path

REPO_ROOT = Path().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
from utils.preview import peek, summary

try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except ImportError:
    pass

DATA = Path('synthetic/v1')
DATA.mkdir(parents=True, exist_ok=True)
(DATA / 'preprocessing').mkdir(exist_ok=True)
(DATA / 'endpoint').mkdir(exist_ok=True)
(DATA / 'decisions').mkdir(exist_ok=True)
(DATA / 'evaluation').mkdir(exist_ok=True)
print(f'Pipeline root: {DATA}')


## 1. Pick source

Real mode: read `dataset/hormones_and_selfreport.csv` and `preprocessing/valid_nights.parquet`, apply the tiered-confidence rule, emit labels. Synthetic mode: demonstrate the schema by reusing the generator's labels and fabricating plausible cycle boundaries.


In [ ]:
REAL_DATA = False

if REAL_DATA and Path('dataset/hormones_and_selfreport.csv').exists():
    print('Using real hormone data from dataset/hormones_and_selfreport.csv')
    source = 'real'
else:
    print('Using synthetic labels (synthetic/v1/labels.parquet) as stand-in for real endpoint.')
    source = 'synthetic'


## 2. Detect cycle boundaries

Real data: per-participant LH-surge detection (local max over a moving window, ≥ threshold) + progesterone-rise check. Synthetic: pretend one cycle per participant with surge at the labelled ovulation day.


In [ ]:
if source == 'synthetic':
    prob = pd.read_parquet(DATA / 'probability_table.parquet')
    lbls = pd.read_parquet(DATA / 'labels.parquet')

    # Recover a plausible LH-surge day per participant: first night_index where
    # p_post_ovulatory crosses 0.5 stably.
    def surge_day(g):
        g = g.sort_values('night_index')
        above = (g['p_post_ovulatory'] >= 0.5).values
        for i in range(len(above) - 1):
            if above[i] and above[i+1]:
                return int(g['night_index'].iloc[i])
        return None

    surges = prob.groupby('participant_id').apply(surge_day, include_groups=False).rename('lh_surge_day').reset_index()
    cov = pd.read_parquet(DATA / 'covariates.parquet')[['participant_id', 'has_round_2']]
    cb = surges.merge(cov, on='participant_id')
    cb['cycle_number'] = 1
    cb['study_interval'] = np.where(cb['has_round_2'], 2024, 2022).astype('int16')
    cb['cycle_start_day'] = 1
    cb['cycle_end_day'] = prob.groupby('participant_id')['night_index'].max().reindex(cb['participant_id']).values
    cb['pdg_rise_confirmed'] = cb['has_round_2'].astype(bool)
    cb['ovulatory'] = cb['lh_surge_day'].notna() & (cb['pdg_rise_confirmed'] | (cb['study_interval'] == 2022))
    cb['confidence_tier'] = np.where(
        cb['pdg_rise_confirmed'] & cb['lh_surge_day'].notna(), 'gold',
        np.where(cb['lh_surge_day'].notna() & (cb['study_interval'] == 2022), 'silver', 'charcoal'))
    cb = cb[['participant_id', 'cycle_number', 'study_interval', 'cycle_start_day',
             'cycle_end_day', 'lh_surge_day', 'pdg_rise_confirmed', 'ovulatory', 'confidence_tier']]
    cb = cb.astype({
        'participant_id': 'string', 'cycle_number': 'int8', 'study_interval': 'int16',
        'cycle_start_day': 'int32', 'cycle_end_day': 'int32',
        'lh_surge_day': 'Int32', 'pdg_rise_confirmed': 'bool', 'ovulatory': 'bool',
        'confidence_tier': 'string',
    })
else:
    raise NotImplementedError('Real-data LH-surge + progesterone detection — TODO for WP3.')

cb.to_parquet(DATA / 'endpoint/cycle_boundaries.parquet', index=False)
peek(DATA / 'endpoint/cycle_boundaries.parquet')


## 3. Apply tiered labels to per-night data

Within the surge window (±2 days) → bronze (0.3). Outside the surge → gold/silver/charcoal inherit from cycle.


In [ ]:
tier_confidence = {'gold': 1.0, 'silver': 0.7, 'bronze': 0.3, 'charcoal': 0.1}

if source == 'synthetic':
    # Reuse generator's label ("pre"/"post") but rewrite label_confidence using the tiered rule.
    lbls2 = lbls.merge(cb[['participant_id', 'lh_surge_day', 'confidence_tier']], on='participant_id')
    near_surge = (lbls2['night_index'] - lbls2['lh_surge_day']).abs() <= 2
    tier_for_night = np.where(near_surge, 'bronze', lbls2['confidence_tier'])
    lbls2['label_confidence'] = pd.Series(tier_for_night, index=lbls2.index).map(tier_confidence).astype('float32')
    lbls2 = lbls2[['participant_id', 'night_index', 'binary_label', 'label_confidence']]
    lbls2 = lbls2.astype({'participant_id': 'string', 'night_index': 'int32',
                          'binary_label': 'string', 'label_confidence': 'float32'})

lbls2.to_parquet(DATA / 'labels.parquet', index=False)
peek(DATA / 'labels.parquet')


## 4. Sanity checks


In [ ]:
from synthetic.validate_contract import validate
validate(DATA)
tier_counts = lbls2['label_confidence'].round(1).value_counts().sort_index()
print('Tier distribution (confidence value → count):')
print(tier_counts)
print(f'Bronze fraction (expected ~20% given ±2 days around ovulation): {(lbls2["label_confidence"] == 0.3).mean():.2%}')


## Next

Open items (see pipeline contract §11):
- Supervisor decision on Option 1 (tiered confidence). When confirmed, move the tier_confidence map into `src/wp3/tiers.py`.
- LH-surge detector spec (local-max threshold multiple, min peak height, smoothing window).
- Progesterone-rise rule (≥2 days > baseline × factor).
- Label-confidence tier → label_confidence mapping (above is a draft).
